In [ ]:
import numpy as np
from scipy import stats
from scipy.optimize import minimize

x_vals = [0, 1, 2, 3, 4] 
bounds = np.array([-np.inf, -1, 0, 1, 2, np.inf])
F_emp = np.array([0.0, 0.1, 0.4, 0.8, 0.9, 1.0])
N = 100
alpha_param = 0.5
sigma_param = 1.2
alpha_level = 0.05

delta_wave = np.sqrt(N) * np.max([
    max(np.abs(stats.norm.cdf(bounds[i], alpha_param, sigma_param) - F_emp[i]),
        np.abs(stats.norm.cdf(bounds[i+1], alpha_param, sigma_param) - F_emp[i+1]))
    for i in range(len(x_vals))
])

bootstrap_iterations = 20000 
bootstrap_delta = np.zeros(bootstrap_iterations)

for i in range(bootstrap_iterations):
    random_sample = np.random.normal(alpha_param, sigma_param, N)
    m_boot, _ = np.histogram(random_sample, bins=bounds)
    
    def neg_log_lik_boot(params):
        mu, sigma = params
        if sigma <= 0: return np.inf
        p_b = stats.norm.cdf(bounds[1:], mu, sigma) - stats.norm.cdf(bounds[:-1], mu, sigma)
        p_b = np.clip(p_b, 1e-12, 1.0)
        return -np.sum(m_boot * np.log(p_b))
    
    boot_guess =[np.mean(random_sample), np.std(random_sample, ddof=1)]
    res_boot = minimize(neg_log_lik_boot, boot_guess, method='Nelder-Mead')
    alpha_boot, sigma_boot = res_boot.x
    
    F_bootstrap_emp = np.array([sum(m_boot[:j]) for j in range(len(m_boot) + 1)]) / N
    
    sup = np.max([
        max(np.abs(stats.norm.cdf(bounds[j], alpha_boot, sigma_boot) - F_bootstrap_emp[j]),
            np.abs(stats.norm.cdf(bounds[j+1], alpha_boot, sigma_boot) - F_bootstrap_emp[j+1]))
        for j in range(len(x_vals))
    ])
    
    bootstrap_delta[i] = np.sqrt(N) * sup

p_value_ks_b = np.sum(bootstrap_delta >= delta_wave) / bootstrap_iterations

print("--- Нормальное распределение (Колмогоров, Параметрический бутстрап) ---")
print(f"Статистика Δ (оригинал) = {delta_wave:.4f}")
print(f"p-value: {p_value_ks_b:.5f}\n")

if p_value_ks_b < alpha_level:
    print(f"p-value ({p_value_ks_b:.5f}) < α ({alpha_level}) ==> H0 ОТВЕРГАЕТСЯ")
else:
    print(f"p-value ({p_value_ks_b:.5f}) >= α ({alpha_level}) ==> Нет оснований отвергать H0")